# Step 8 – Fine-tune on Nepal (research question 2)
**Idea:** on 13 Oct 2024 Sentinel-1 (orbit 12) and Sentinel-2 imaged the Koshi area on the same day. Sentinel-2 water labels (MNDWI, clear pixels only) become training labels for the matching Sentinel-1 image.

**Fair test design**
- Training uses only the **north half** of the Koshi box (13 Oct).
- Testing uses only the **south half**, on the **1 Oct flood image**, scored against the 8 Oct Sentinel-2 labels (as in notebook 06).
- A 64-pixel gap separates the halves. Different dates and different places: no leakage.

Models A and B (3 seeds each, from notebooks 02/05) are fine-tuned and compared with their original versions. The fine-tuned models are also re-checked on **Kathmandu**, to see whether adapting to flat Koshi changes false alarms in the mountains.

Runtime: **GPU (T4)**, about 30–40 min. Needs Earth Engine and Drive.

In [ ]:
PROJECT = 'your-ee-project-id'   # <-- your Earth Engine Cloud project ID
!pip -q install segmentation-models-pytorch rasterio
import os, glob, math, time, random, subprocess, numpy as np, pandas as pd, rasterio, torch, torch.nn as nn, ee
import segmentation_models_pytorch as smp, matplotlib.pyplot as plt
from rasterio.warp import reproject, Resampling
from scipy.ndimage import binary_erosion
from torch.utils.data import Dataset, DataLoader
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEV)

from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Paper2_flood'
NEPAL = '/content/drive/MyDrive/Paper2_flood_nepal'
OUT = f'{BASE}/08_finetune'; os.makedirs(OUT, exist_ok=True)
F = {'koshi': f'{NEPAL}/koshi_terai_flood2024-10-01_pre2024-09-19_orbit12.tif',
     'ktm': f'{NEPAL}/kathmandu_flood2024-10-02_pre2024-09-20_orbit19.tif',
     'labels_oct8': f'{NEPAL}/koshi_terai_s2labels_2024-10-08.tif'}
for k, f in F.items(): assert os.path.exists(f), f'missing {f}'
WEIGHTS = {('A_sar', 0): f'{BASE}/02_terrain/unet_A_sar.pt', ('B_sar_terrain', 0): f'{BASE}/02_terrain/unet_B_sar_terrain.pt'}
for m in ['A_sar', 'B_sar_terrain']:
    for s in [1, 2]: WEIGHTS[(m, s)] = f'{BASE}/05_multiseed/runs/unet_{m}_seed{s}.pt'
for k, f in WEIGHTS.items(): assert os.path.exists(f), f'missing model {f}'
CH = {'A_sar': [0, 1], 'B_sar_terrain': [0, 1, 2, 3]}

ee.Authenticate(); ee.Initialize(project=PROJECT)
AOI = ee.Geometry.Rectangle([86.85, 26.45, 87.15, 26.75])
print('ready')

## 1. Export the 13 Oct training pair (S1 VV, VH + S2 water label)

In [ ]:
TRAIN_DATE = '2024-10-13'
F['train'] = f'{NEPAL}/koshi_terai_train_s1s2_{TRAIN_DATE}.tif'
if not os.path.exists(F['train']):
    d = ee.Date(TRAIN_DATE)
    s1c = (ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(AOI).filterDate(d, d.advance(1, 'day'))
           .filter(ee.Filter.eq('instrumentMode', 'IW')).filter(ee.Filter.eq('relativeOrbitNumber_start', 12))
           .select(['VV', 'VH']))
    print('S1 scenes on', TRAIN_DATE, ':', s1c.size().getInfo())
    s1 = s1c.mosaic().setDefaultProjection(s1c.first().select('VV').projection())
    s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(AOI).filterDate(d, d.advance(1, 'day')).mosaic()
    scl = s2.select('SCL'); clear = scl.eq(2).Or(scl.gte(4).And(scl.lte(7)))
    nd = s2.normalizedDifference(['B3', 'B11'])
    lab = ee.Image(255).where(nd.gt(0.1), 1).where(nd.lt(-0.1), 0).updateMask(clear).unmask(255)
    img = ee.Image.cat([s1.toFloat(), lab.toFloat().rename('s2_water')]).clip(AOI)
    task = ee.batch.Export.image.toDrive(image=img, description='koshi_train_pair', folder='Paper2_flood_nepal',
            fileNamePrefix=f'koshi_terai_train_s1s2_{TRAIN_DATE}', region=AOI, scale=10, crs='EPSG:32645', maxPixels=1e10)
    task.start()
    while task.status()['state'] in ('READY', 'RUNNING'):
        print(task.status()['state']); time.sleep(30)
    print('export:', task.status()['state'], task.status().get('error_message', ''))
    for _ in range(20):
        if os.path.exists(F['train']): break
        time.sleep(15)
assert os.path.exists(F['train']), 'training file not in Drive yet - wait a minute and rerun this cell'
print('training pair ready')

## 2. Load everything on the Koshi grid; define north (train) and south (test) halves

In [ ]:
BANDS = ['flood_VV', 'flood_VH', 'pre_VV', 'pre_VH', 'slope', 'hand', 'perm_water', 'ls_mask', 'quick_flood']
def read_stack(path):
    with rasterio.open(path) as s: return dict(zip(BANDS, s.read().astype('float32'))), s.profile
kb, kprof = read_stack(F['koshi']); shape = kb['flood_VV'].shape

def on_grid(path, n, fill, prof=kprof, shp=None):
    shp = shp or shape
    out = np.full((n,) + shp, fill, 'float32')
    with rasterio.open(path) as src:
        for i in range(n):
            reproject(rasterio.band(src, i + 1), out[i], dst_transform=prof['transform'], dst_crs=prof['crs'],
                      resampling=Resampling.nearest)
    return out

tr = on_grid(F['train'], 3, 255)                    # VV, VH, label on 13 Oct
lab8 = on_grid(F['labels_oct8'], 2, 255)            # s2_water, dry_water on 8 Oct

def valid_of(*arrs):
    v = np.ones(shape, bool)
    for a in arrs: v &= np.isfinite(a) & (a != 0)
    return binary_erosion(v, iterations=3)
v_test = valid_of(kb['flood_VV'], kb['flood_VH'])
v_train = valid_of(tr[0], tr[1])

H = shape[0]; GAP = 64
rows = np.arange(H)[:, None]
north = np.broadcast_to(rows < H // 2 - GAP // 2, shape)
south = np.broadcast_to(rows >= H // 2 + GAP // 2, shape)
train_lab = np.where(v_train & north, tr[2], 255).astype('int64')
print(f"train labels: {(train_lab != 255).sum() / 1e4:.0f} km² labelled, "
      f"water share {(train_lab[train_lab != 255] == 1).mean() * 100:.1f}%")

# JRC normal water (occurrence >= 80 %) on the same grid
t = kprof['transform']
req = {'expression': ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').unmask(0).toUint8(),
       'fileFormat': 'NUMPY_NDARRAY',
       'grid': {'dimensions': {'width': shape[1], 'height': shape[0]}, 'crsCode': kprof['crs'].to_string(),
                'affineTransform': {'scaleX': t.a, 'shearX': t.b, 'translateX': t.c, 'shearY': t.d, 'scaleY': t.e, 'translateY': t.f}}}
normal = ee.data.computePixels(req)['occurrence'] >= 80
print('JRC loaded')

## 3. Input scaling (same as training) and helpers

In [ ]:
ROOT = '/content/sen1floods11'; os.makedirs(ROOT, exist_ok=True)
if not os.path.exists(f'{ROOT}/S1Hand'):
    for p in ['data/flood_events/HandLabeled/S1Hand', 'splits/flood_handlabeled']:
        subprocess.run(['gsutil', '-m', '-q', 'cp', '-r', f'gs://sen1floods11/v1.1/{p}', f'{ROOT}/'], check=True)
CACHE = f'{BASE}/02_terrain/terrain_chips'

def prep(sar, slope, hand):
    sar = np.clip(np.nan_to_num(sar, nan=-50), -50, 1)
    t = np.stack([np.clip(np.nan_to_num(slope), 0, 60), np.log1p(np.clip(np.nan_to_num(hand), 0, 200))])
    return np.concatenate([sar, t]).astype('float32')

ids = [f.replace('_S1Hand.tif', '') for f in pd.read_csv(f'{ROOT}/flood_handlabeled/flood_train_data.csv', header=None)[0]]
xs = []
for c in ids:
    with rasterio.open(f'{ROOT}/S1Hand/{c}_S1Hand.tif') as s: sar = s.read().astype('float32')
    tt = np.load(f'{CACHE}/{c}.npy'); xs.append(prep(sar, tt[0], tt[1]))
xs = np.stack(xs); MEAN, STD = xs.mean((0, 2, 3)), xs.std((0, 2, 3)) + 1e-6; del xs

def norm(x, valid):
    x = (x - MEAN[:, None, None]) / STD[:, None, None]; x[:, ~valid] = 0; return x

X_train = norm(prep(tr[:2], kb['slope'], kb['hand']), v_train)
X_flood = norm(prep(np.stack([kb['flood_VV'], kb['flood_VH']]), kb['slope'], kb['hand']), v_test)

def new_model(name):
    return smp.Unet('resnet34', encoder_weights=None, in_channels=len(CH[name]), classes=2).to(DEV)

@torch.no_grad()
def predict(model, x, tile=512, stride=448, batch=8):
    model.eval(); C_, H_, W_ = x.shape
    Hp = max(tile, math.ceil((H_ - tile) / stride) * stride + tile); Wp = max(tile, math.ceil((W_ - tile) / stride) * stride + tile)
    xp = np.pad(x, ((0, 0), (0, Hp - H_), (0, Wp - W_)), mode='edge')
    prob = np.zeros((Hp, Wp), 'float32'); cnt = np.zeros((Hp, Wp), 'float32')
    pos = [(r, c) for r in range(0, Hp - tile + 1, stride) for c in range(0, Wp - tile + 1, stride)]
    for i in range(0, len(pos), batch):
        pb = pos[i:i + batch]
        p = torch.softmax(model(torch.tensor(np.stack([xp[:, r:r+tile, c:c+tile] for r, c in pb])).to(DEV)), 1)[:, 1].cpu().numpy()
        for (r, c), pi in zip(pb, p): prob[r:r+tile, c:c+tile] += pi; cnt[r:r+tile, c:c+tile] += 1
    return (prob / cnt)[:H_, :W_] > 0.5

def score(pred, truth, mask):
    p, tt = pred[mask], truth[mask]
    tp, fp, fn = (p & tt).sum(), (p & ~tt).sum(), (~p & tt).sum()
    pr, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return dict(IoU=tp / max(tp + fp + fn, 1), precision=pr, recall=rc)

# test truth (south half only)
s2w, dryw = lab8[0], lab8[1]
m1 = v_test & south & (s2w != 255); truth1 = s2w == 1
m2 = m1 & (dryw != 255); truth2 = (s2w == 1) & (dryw == 0)
print(f'test area: water extent {m1.sum() / 1e4:.0f} km², new water {m2.sum() / 1e4:.0f} km²')

## 4. Fine-tune
Random 256 × 256 crops from the north half with at least 30% labelled pixels, flips, learning rate 10⁻⁴, 10 epochs × 400 crops. Same loss as the original training.

In [ ]:
class NorthCrops(Dataset):
    def __init__(self, ch, n=400, crop=256):
        self.ch, self.n, self.crop = ch, n, crop
        self.rmax = H // 2 - GAP // 2 - crop; self.cmax = shape[1] - crop
    def __len__(self): return self.n
    def __getitem__(self, i):
        for _ in range(50):
            r, c = np.random.randint(0, self.rmax), np.random.randint(0, self.cmax)
            y = train_lab[r:r+self.crop, c:c+self.crop]
            if (y != 255).mean() >= 0.3: break
        x = X_train[self.ch, r:r+self.crop, c:c+self.crop]
        if random.random() < .5: x, y = x[:, :, ::-1], y[:, ::-1]
        if random.random() < .5: x, y = x[:, ::-1], y[::-1]
        return torch.tensor(x.copy()), torch.tensor(y.copy())

def finetune(name, seed, epochs=10):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    m = new_model(name); m.load_state_dict(torch.load(WEIGHTS[(name, seed)], map_location=DEV))
    dl = DataLoader(NorthCrops(CH[name]), batch_size=16, shuffle=True, num_workers=0)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1., 3.], device=DEV), ignore_index=255)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=1e-4)
    for ep in range(epochs):
        m.train(); tot = 0
        for x, y in dl:
            opt.zero_grad(); l = loss_fn(m(x.to(DEV)), y.to(DEV)); l.backward(); opt.step(); tot += l.item()
        if ep in (0, epochs - 1): print(f'[{name} seed {seed}] epoch {ep} loss {tot / len(dl):.3f}')
    torch.save(m.state_dict(), f'{OUT}/unet_{name}_seed{seed}_finetuned.pt')
    return m

## 5. Evaluate original vs fine-tuned on the south half (1 Oct flood vs 8 Oct Sentinel-2)

In [ ]:
kt, kprof_ = read_stack(F['ktm']); kshape = kt['flood_VV'].shape
kv = np.ones(kshape, bool)
for k in ['flood_VV', 'flood_VH', 'pre_VV', 'pre_VH']: kv &= np.isfinite(kt[k]) & (kt[k] != 0)
kv = binary_erosion(kv, iterations=3)
def kn(x):
    x = (x - MEAN[:, None, None]) / STD[:, None, None]; x[:, ~kv] = 0; return x
K_flood = kn(prep(np.stack([kt['flood_VV'], kt['flood_VH']]), kt['slope'], kt['hand']))
K_pre = kn(prep(np.stack([kt['pre_VV'], kt['pre_VH']]), kt['slope'], kt['hand']))
kperm, kls = kt['perm_water'] > 0, kt['ls_mask'] > 0

rows = []
for name in ['A_sar', 'B_sar_terrain']:
    for seed in [0, 1, 2]:
        base = new_model(name); base.load_state_dict(torch.load(WEIGHTS[(name, seed)], map_location=DEV))
        tuned = finetune(name, seed)
        for version, m in [('original', base), ('fine-tuned', tuned)]:
            w = predict(m, X_flood[CH[name]])
            l1 = score(w, truth1, m1); l2 = score(w & ~normal, truth2, m2)
            kf = predict(m, K_flood[CH[name]]) & ~predict(m, K_pre[CH[name]]) & ~kperm & kv
            rows.append({'model': name, 'seed': seed, 'version': version,
                         'L1 water IoU': l1['IoU'], 'L1 precision': l1['precision'], 'L1 recall': l1['recall'],
                         'L2 new-water IoU': l2['IoU'], 'L2 precision': l2['precision'], 'L2 recall': l2['recall'],
                         'KTM false new water km²': kf.sum() / 1e4, 'KTM in layover/shadow km²': (kf & kls).sum() / 1e4})
        print('done', name, seed)
res = pd.DataFrame(rows); res.to_csv(f'{OUT}/finetune_results_per_seed.csv', index=False)

base_l1 = score(kb['flood_VV'] < -15, truth1, m1); base_l2 = score((kb['flood_VV'] < -15) & ~normal, truth2, m2)
print(f"threshold baseline (south half): L1 IoU {base_l1['IoU']:.3f}, L2 IoU {base_l2['IoU']:.3f}")

## 6. Summary: mean ± std over 3 seeds

In [ ]:
cols = ['L1 water IoU', 'L1 precision', 'L1 recall', 'L2 new-water IoU', 'L2 precision', 'L2 recall',
        'KTM false new water km²', 'KTM in layover/shadow km²']
g = res.groupby(['model', 'version'])[cols]
summary = (g.mean().round(3).astype(str) + ' ± ' + g.std().round(3).astype(str)).T
summary.to_csv(f'{OUT}/finetune_summary.csv'); summary

In [ ]:
w = res.pivot_table(index=['model', 'seed'], columns='version', values=['L1 water IoU', 'L2 new-water IoU', 'KTM false new water km²'])
gain = pd.DataFrame({k: w[(k, 'fine-tuned')] - w[(k, 'original')] for k in ['L1 water IoU', 'L2 new-water IoU', 'KTM false new water km²']})
gsum = gain.groupby('model').agg(['mean', 'std']).round(3)
gsum['seeds L2 improved'] = gain.groupby('model')['L2 new-water IoU'].apply(lambda x: f'{(x > 0).sum()}/{len(x)}')
gsum.to_csv(f'{OUT}/finetune_gain.csv'); gsum

## 7. Map: model B seed 0, south half, new water

In [ ]:
name = 'B_sar_terrain'
b0 = new_model(name); b0.load_state_dict(torch.load(WEIGHTS[(name, 0)], map_location=DEV))
t0 = new_model(name); t0.load_state_dict(torch.load(f'{OUT}/unet_{name}_seed0_finetuned.pt', map_location=DEV))
def rgb(pred, truth, mask):
    im = np.full(shape + (3,), 0.85); im[mask] = 1
    im[mask & pred & truth] = [0.1, 0.6, 0.1]; im[mask & pred & ~truth] = [0.85, 0.1, 0.1]; im[mask & ~pred & truth] = [0.1, 0.3, 0.9]
    return im[H // 2::3, ::3]
fig, ax = plt.subplots(1, 2, figsize=(14, 7))
for a, (t_, m) in zip(ax, [('original', b0), ('fine-tuned', t0)]):
    a.imshow(rgb(predict(m, X_flood[CH[name]]) & ~normal, truth2, m2)); a.set_title(f'B {t_}: new water, south half'); a.axis('off')
fig.suptitle('Green correct, red false alarm, blue missed, grey no label')
plt.tight_layout(); plt.savefig(f'{OUT}/finetune_map_B.png', dpi=110)